# SeoulMate RAG + Weather MCP 디버그 노트북

최종 GPT 답변 이전의 전체 과정을 확인합니다.

1. Structured Query와 Restaurant Search Plan
2. RAG에 들어가는 실제 검색 문장과 필터
3. 원본 RAG 후보 최대 30개의 전체 순위와 점수 구성
4. 각 후보의 리뷰·메뉴 근거
5. Weather MCP 원본 응답
6. 날씨 적용 후 전체 순위, 순위 변화, 가감점 이유
7. 선택 후보 상세 근거

`QUESTION_CASE` 셀만 바꾸면 다른 질문도 같은 방식으로 검사할 수 있습니다.

In [1]:
# 1. 절대 경로 기준 실행 환경
import os
import sys
from copy import deepcopy
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

import html
import json
from IPython.display import HTML, display

BACKEND_DIR = Path(r"C:\Users\user\Desktop\seoulmate\SeoulMate\backend")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)

def show_table(rows):
    """pandas 없이 list[dict]를 가로 스크롤 가능한 HTML 표로 표시합니다."""
    rows = list(rows)
    if not rows:
        display(HTML("<em>표시할 행이 없습니다.</em>"))
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    def text(value):
        if isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False, default=str)
        return html.escape(str(value))
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{text(row.get(column, ''))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<div style='overflow-x:auto;max-width:100%'>"
        "<table style='border-collapse:collapse;font-size:12px'>"
        "<style>th,td{border:1px solid #bbb;padding:5px;vertical-align:top;white-space:nowrap}"
        "td{max-width:420px;white-space:normal}</style>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))

print("backend:", BACKEND_DIR)
print("python:", sys.executable)

backend: C:\Users\user\Desktop\seoulmate\SeoulMate\backend
python: C:\Users\user\anaconda3\python.exe


## 2. 테스트 질문과 고정 Structured Query

실서비스에서는 상위 GPT가 이 JSON을 만듭니다. 여기서는 GPT 파싱 결과의 변동과 비용을 분리하기 위해 JSON을 직접 넣습니다.

In [2]:
# 이 셀을 수정해서 여러 질문을 테스트하세요.
QUESTION_CASE = {
    "language": "ko",
    "intent": "single_place_recommendation",
    "original_question": "내일 저녁 홍대에서 조용하고 분위기 좋은 식당 추천해줘",
    "normalized_question": "내일 저녁 홍대의 조용하고 분위기 좋은 식당 추천",
    "tasks": [{
        "task_id": "task_1",
        "domain": "restaurant",
        "search_query": "홍대 조용하고 분위기 좋은 식당",
        "themes": ["조용한", "분위기 좋은", "저녁"],
        "desired_count": 3,
        "notes": None,
    }],
    "filters": {
        "location": "홍대",
        "radius_km": None,
        "is_active": True,
        "start_date": "2026-07-15",
        "end_date": "2026-07-15",
        "time_window": "evening",
        "party_size": None,
        "budget_min_krw": None,
        "budget_max_krw": None,
        "transportation": [],
        "accessibility": [],
        "required_features": [],
        "excluded_features": [],
    },
    "weather_request": {
        "query": "내일 저녁 홍대 날씨",
        "location_name": "홍대",
        "target_date": "2026-07-15",
        "target_time": "evening",
        "language": "ko",
    },
    "general_response_instruction": None,
}

# 사용자 현재 위치가 서울시청이어도 filters.location=홍대를 우선해야 하는 상황
CURRENT_LAT = 37.5665
CURRENT_LNG = 126.9780
CURRENT_LOCATION_NAME = None
TOP_N = 30

pprint(QUESTION_CASE)

{'filters': {'accessibility': [],
             'budget_max_krw': None,
             'budget_min_krw': None,
             'end_date': '2026-07-15',
             'excluded_features': [],
             'is_active': True,
             'location': '홍대',
             'party_size': None,
             'radius_km': None,
             'required_features': [],
             'start_date': '2026-07-15',
             'time_window': 'evening',
             'transportation': []},
 'general_response_instruction': None,
 'intent': 'single_place_recommendation',
 'language': 'ko',
 'normalized_question': '내일 저녁 홍대의 조용하고 분위기 좋은 식당 추천',
 'original_question': '내일 저녁 홍대에서 조용하고 분위기 좋은 식당 추천해줘',
 'tasks': [{'desired_count': 3,
            'domain': 'restaurant',
            'notes': None,
            'search_query': '홍대 조용하고 분위기 좋은 식당',
            'task_id': 'task_1',
            'themes': ['조용한', '분위기 좋은', '저녁']}],
 'weather_request': {'language': 'ko',
                     'location_name': '홍대',
             

In [3]:
# 3. Structured Query -> 실제 Restaurant Search Plan
from schemas.structured_query import StructuredTravelQuery
from services.query_policy import derive_source_mode
from services.rag import build_restaurant_search_plan

parsed_query = StructuredTravelQuery.model_validate(QUESTION_CASE)
restaurant_task = next(task for task in parsed_query.tasks if task.domain == "restaurant")
source_mode = derive_source_mode(parsed_query)
plan = build_restaurant_search_plan(
    parsed_query,
    restaurant_task,
    current_lat=CURRENT_LAT,
    current_lng=CURRENT_LNG,
    current_location_name=CURRENT_LOCATION_NAME,
    top_n=TOP_N,
)

print("source_mode:", source_mode)
show_table({"field": key, "value": value} for key, value in asdict(plan).items())

source_mode: rag_mcp


field,value
task_id,task_1
retrieval_query,조용하고 분위기 좋은 식당 조용한 저녁
review_query,조용하고 분위기 좋은 식당 조용한 저녁
menu_query,조용하고 분위기 좋은 식당
requested_category,None
location_name,홍대입구역 2호선
origin_lat,37.5568707448873
origin_lng,126.923778562273
radius_km,2.0
open_now,False


### RAG가 실제 사용하는 값

- `retrieval_query`: 식당·메뉴 임베딩 검색 문장
- `review_query`: 음식 종류를 제거한 리뷰 임베딩 검색 문장
- `requested_category`: 명시적 음식 종류 하드 조건
- `origin_lat/lng`, `radius_km`: DB 후보 공간 범위
- `target_visit_at`: 영업시간 판정 시각
- `required_feature_fields`, `budget_*`: 후보 생성 단계 하드 필터

In [4]:
# 4. RAG 실행 전 입력 요약
rag_input = {
    "task_id": plan.task_id,
    "language_table": "en" if parsed_query.language.lower().startswith("en") else "ko",
    "retrieval_query": plan.retrieval_query,
    "review_query": plan.review_query,
    "requested_category": plan.requested_category,
    "resolved_location": plan.location_name,
    "origin_lat": plan.origin_lat,
    "origin_lng": plan.origin_lng,
    "radius_km": plan.radius_km,
    "open_now": plan.open_now,
    "target_visit_at": plan.target_visit_at,
    "min_rating": plan.min_rating,
    "budget_min_krw": plan.budget_min_krw,
    "budget_max_krw": plan.budget_max_krw,
    "required_feature_fields": plan.required_feature_fields,
    "excluded_feature_fields": plan.excluded_feature_fields,
    "include_weather_features": plan.include_weather_features,
    "top_n": plan.top_n,
}
show_table([rag_input])

task_id,language_table,retrieval_query,review_query,requested_category,resolved_location,origin_lat,origin_lng,radius_km,open_now,target_visit_at,min_rating,budget_min_krw,budget_max_krw,required_feature_fields,excluded_feature_fields,include_weather_features,top_n
task_1,ko,조용하고 분위기 좋은 식당 조용한 저녁,조용하고 분위기 좋은 식당 조용한 저녁,None,홍대입구역 2호선,37.5568707448873,126.923778562273,2.0,False,2026-07-16 19:00:00,None,None,None,[],[],True,30


In [6]:
# 5. 원본 RAG 검색 실행 (MCP 재랭킹 전)
from services.rag import search_restaurants_structured

rag_result = search_restaurants_structured(
    parsed_query,
    restaurant_task,
    current_lat=CURRENT_LAT,
    current_lng=CURRENT_LNG,
    current_location_name=CURRENT_LOCATION_NAME,
    top_n=TOP_N,
)
raw_candidates = deepcopy(rag_result["candidates"])

print("검색 위치:", rag_result.get("location_name"), rag_result.get("origin_lat"), rag_result.get("origin_lng"))
print("추출 음식 종류:", rag_result.get("extracted_category"))
print("RAG 후보 수:", len(raw_candidates))

검색 위치: 홍대입구역 2호선 37.5568707448873 126.923778562273
추출 음식 종류: None
RAG 후보 수: 30


In [7]:
# 6. MCP 적용 전 전체 RAG 순위와 점수 구성
def compact_reviews(candidate):
    return " | ".join(
        f"{r.get('similarity', 0):.3f}:{str(r.get('content', ''))[:80]}"
        for r in candidate.get("evidence", {}).get("reviews", [])
    )

def compact_menus(candidate):
    return " | ".join(
        f"{m.get('similarity', 0):.3f}:{m.get('menu_name', '')}"
        for m in candidate.get("evidence", {}).get("menus", [])
    )

def rag_rows(candidates):
    rows = []
    for rank, c in enumerate(candidates, 1):
        b = c.get("breakdown", {})
        rows.append({
            "rag_rank": rank,
            "restaurant_id": c["restaurant_id"],
            "name": c["name"],
            "category": c.get("category"),
            "category_kakao": c.get("category_kakao"),
            "category_status": b.get("category_status"),
            "category_confidence": b.get("category_confidence"),
            "rag_score": c.get("score"),
            "restaurant_rrf": b.get("restaurant_rrf"),
            "review_rrf": b.get("review_rrf"),
            "menu_rrf": b.get("menu_rrf"),
            "base_score": b.get("base_score"),
            "category_boost": b.get("category_boost"),
            "rating_preference_boost": b.get("rating_preference_boost"),
            "distance_km": c.get("distance_km"),
            "rating": c.get("rating"),
            "review_count": c.get("review_count"),
            "open_at_requested_time": c.get("open_status"),
            "open_status_basis": c.get("open_status_basis"),
            "menu_price_median": c.get("menu_price_median"),
            "parking": c.get("has_parking"),
            "pets": c.get("allows_pets"),
            "group_seating": c.get("has_group_seating"),
            "private_room": c.get("has_private_room"),
            "top_review_evidence": compact_reviews(c),
            "top_menu_evidence": compact_menus(c),
        })
    return rows

rag_rows_data = rag_rows(raw_candidates)
show_table(rag_rows_data)

rag_rank,restaurant_id,name,category,category_kakao,category_status,category_confidence,rag_score,restaurant_rrf,review_rrf,menu_rrf,base_score,category_boost,rating_preference_boost,distance_km,rating,review_count,open_at_requested_time,open_status_basis,menu_price_median,parking,pets,group_seating,private_room,top_review_evidence,top_menu_evidence
1,33405834,살롱 드 상상,프랑스 요리,None,n/a,None,0.10642024113732206,0.010606060606060605,0.09581418053126145,0.0,0.10642024113732206,0.0,0.0,1.1692589521459695,5.0,22,True,requested_time,None,None,None,None,None,0.656:분위기가 너무 좋고 맛있어요! 조용한 분위기라 대화하기도 좋고 무엇보다 음식이 맛잇네요 ㅎㅎ 게다가 직원분들도 너무 친절해서 좋은 식사였습니다! | 0.632:맛있고 분위기도 좋음!! 클래식 음악 잔잔하게 나와서 차분하게 대화하면서 먹었어요 점심식사하기도 좋고 저녁에 와인이랑 먹어도 좋을거같아여 | 0.610:음악이랑 인테리어가 좋습니다 조용히 대화나누기도 좋고 무엇보다 음식이 하나하나 신선하고 맛있어요 재방문 예정입니다,
2,25453046,Hangong-Gan,"아시아 요리, 한국, 퓨전",퓨전요리,n/a,None,0.10197725822568388,0.00813953488372093,0.08150038300454632,0.012337340337416626,0.10197725822568388,0.0,0.0,0.5505386241093928,5.0,278,True,requested_time,23000,True,None,None,None,0.571:예약해서 다녀왔는데 색다른 메뉴라 신선하고 맛있고 분위기도 좋아요 | 0.554:너무 아늑하고 조용하고 프라이빗한 분위기여서 너무 좋았어요! 음식도 정말 너무 맛있게 먹었어요 없던 입맛도 한 입 먹고 바로 생겨서 한 톨도 남 | 0.532:예상치 못한 맛이라서 좋았어요. 분위기도 너무 좋네요. 예약 필수일듯 합니다.,0.373:푸주 마라 순살 닭구이 | 0.372:부추 소금 순살 닭구이 | 0.350:고추장 직화 순살 닭구이
3,6538765,루블랑,"프랑스 요리, 스테이크하우스, 유럽 요리, 와인 바",None,n/a,None,0.09484203729245018,0.01,0.07605320065687853,0.008788836635571659,0.09484203729245018,0.0,0.0,0.36205006973087395,4.4,55,True,requested_time,29000,True,None,None,None,"0.538:이 지하 를 자랑하는 세련된 된 조용한 맛있는 음식과. 양쪽에 있는 것이 바로 entrée 분할 및 점심 식사. 비싼 기대에 기반하지 않은 리뷰 | 0.536:식사는 67 kw 2, 서울 에 비해 다른 프렌치 레스토랑. 좋은 품질과 수량. 좋은 분위기, 인테리어 & 음악. | 0.528:약간 위치가 찾기 어려웠지만 맛있는 프랑스 가정식을 맛볼 수 있는곳이었어요. 분위기도 좋고 직원분들도 친절했습니다. 데이트 코스로도 좋은거 같아",0.374:수비드 삼겹살 | 0.369:생면 라구 스파게티
4,9598657,랑빠스81,"프랑스 요리, 유럽 요리, 와인 바",양식,n/a,None,0.08984681665252393,0.01129032258064516,0.07100431348347261,0.0075521805884061575,0.08984681665252393,0.0,0.0,0.38436111312037236,4.9,98,True,requested_time,28000,False,None,None,None,"0.547:친절한 직원, 쿨한 인테리어 , 맛있는 음식 | 0.519:연남동에서 근사한 식사를 하고 싶으시다면 꼭 가보세요!! 가게 분위기도 너무 좋고 미슐랭 선정까지 되었으니 음식맛은 당연 최고입니다👍🏼👍🏼 음식 | 0.518:결혼기념일에 남편이 특별히 예약해서 갔어요. 와인과 완벽한 마리아주, 디저트까지 너무 좋았어요. 분위기도 음식맛에 영향을 줄 만큼 특별했어요",0.400:건자두 삼겹살과 감자퓨레 | 0.299:에스카르고 샐러드
5,8728506,미분당 신촌점,None,베트남음식,n/a,None,0.0852650178452127,0.011475409836065573,0.059126741271784106,0.014662866737363027,0.0852650178452127,0.0,0.0,0.864067545614737,4.7,58,True,requested_time,None,False,None,None,None,"0.526:쌀국수 정말 맛있었어요. 가격도 저렴한 편이구요. 적당히 조용한 건 좋은데 너무너무 조용해서 솔직히 부담스럽더라구요. 혼자 가볍게 식사하고 싶을 | 0.513:1. 음식 향신료 잘 못드시는 분들도 맛있게 먹을 수 있을 것 같아요. 한국인 입맛에 맞춘 쌀국수 느낌? 국물은 짜지않고 맑은 느낌이고, 해선장 | 0.487:신촌에서 쌀국수 하면 무조건 미분당을 추천합니다. 자리도 1인석이 많고 식당 안에서 조용히 식사하여야하는 규칙 덕에 혼밥하기도 최적화되어 있습니",0.404:차돌양지 쌀국수 | 0.371:차돌양지힘줄 쌀국수 | 0.367:차돌박이 쌀국수
6,1139517,벽제 갈비,"바베큐, 아시아 요리, 한국, 그릴",갈비,n/a,None,0.08399727182997938,0.008235294117647058,0.056577681577681574,0.019184296134650745,0.08399727182997938,0.0,0.0,1.2416434839847603,4.3,68,True,requested_time,32000,True,None,True,True,"0.511:우수한 친절한 서비스, 맛있는 음식과 멋진 경험! | 0.500:식당 내부가 넓고 깔끔하고 음식과 밑반찬들도 맛있었어요 가족모임이나 회식장소로도 괜찮을 것 같아요 가격대가 높을만 해요 | 0.495:흠잡을 곳 없는 맛의 밸런스를 가진 한우를 먹을 수 있는 곳. 가격대가 비싸긴 하지만 그에 맞는 서비스와 맛을 즐길 수 있다. 조용한 곳을 원한",0.430:한우불고기정식 | 0.403:한우갈비탕 | 0.390:한우불고기냉면정식
7,25139751,코스모 더 케이브,"이탈리아 요리, 아시아 요리, 한국, 퓨전, 와인 바",None,n/a,None,0.07787079864323157,0.0076086956521739125,0.07026210299105766,0.0,0.07787079864323157,0.0,0.0,0.3844765878972961,5.0,11,True,requested_time,None,None,True,None,None,"0.570:야외 테라스에서 마시는 내추럴와인은 정말 환상적이었음. 선선한 바람과 안에서 흘러나오는 노래, 감각적인 조명과 꽃장식이 최고의 조합이었다. 연남 | 0.517:여행하다 숨겨진 맛집 찾은 느낌 🙋🏻‍♀️ 이건 리뷰 남겨야함 감각적인 분위기 맛있는 음식 내추럴와인 다양하고 잔와인도 있어서 좋음 날씨 좋은날 | 0.510:보통은 음악 듣고 잔와인 마시고 싶을 때 부담없이 가는 플레이스에요. 아주 가끔 날씨 좋을 땐 가족끼리 가서 마당에 아이랑 놀면서 바틀이랑 음식",
8,26379212,온달집 홍대직영점,"아시아 요리, 한국, 델리, 그릴, 펍, 퓨전",None,n/a,None,0.0777092242180033,0.007692307692307692,0.07001691652569561,0.0,0.0777092242180033,0.0,0.0,0.7173604637217327,5.0,49,True,requested_time,None,None,No

In [8]:
# 7. 후보별 리뷰/메뉴 근거를 행 단위로 펼쳐서 확인
evidence_rows = []
for rank, candidate in enumerate(raw_candidates, 1):
    for review in candidate.get("evidence", {}).get("reviews", []):
        evidence_rows.append({
            "rag_rank": rank,
            "restaurant_id": candidate["restaurant_id"],
            "name": candidate["name"],
            "evidence_type": "review",
            "global_vector_rank": review.get("rank"),
            "similarity": review.get("similarity"),
            "is_main": None,
            "content": review.get("content"),
        })
    for menu in candidate.get("evidence", {}).get("menus", []):
        evidence_rows.append({
            "rag_rank": rank,
            "restaurant_id": candidate["restaurant_id"],
            "name": candidate["name"],
            "evidence_type": "menu",
            "global_vector_rank": menu.get("rank"),
            "similarity": menu.get("similarity"),
            "is_main": menu.get("is_main"),
            "content": menu.get("menu_name"),
        })

show_table(evidence_rows)

rag_rank,restaurant_id,name,evidence_type,global_vector_rank,similarity,is_main,content
1,33405834,살롱 드 상상,review,1,0.6563918073163097,None,분위기가 너무 좋고 맛있어요! 조용한 분위기라 대화하기도 좋고 무엇보다 음식이 맛잇네요 ㅎㅎ 게다가 직원분들도 너무 친절해서 좋은 식사였습니다!
1,33405834,살롱 드 상상,review,2,0.6324943638509558,None,맛있고 분위기도 좋음!! 클래식 음악 잔잔하게 나와서 차분하게 대화하면서 먹었어요 점심식사하기도 좋고 저녁에 와인이랑 먹어도 좋을거같아여
1,33405834,살롱 드 상상,review,5,0.609620041420569,None,음악이랑 인테리어가 좋습니다 조용히 대화나누기도 좋고 무엇보다 음식이 하나하나 신선하고 맛있어요 재방문 예정입니다
2,25453046,Hangong-Gan,review,8,0.5706542864760609,None,예약해서 다녀왔는데 색다른 메뉴라 신선하고 맛있고 분위기도 좋아요
2,25453046,Hangong-Gan,review,13,0.5542538185984358,None,너무 아늑하고 조용하고 프라이빗한 분위기여서 너무 좋았어요! 음식도 정말 너무 맛있게 먹었어요 없던 입맛도 한 입 먹고 바로 생겨서 한 톨도 남기지 않고 다 만족스럽게 먹었습니다! 맥주도 너무 부드러워서 새로웠어요 조만간 재방문 오겠습니다🙏🏻
2,25453046,Hangong-Gan,review,21,0.5315587984638909,None,예상치 못한 맛이라서 좋았어요. 분위기도 너무 좋네요. 예약 필수일듯 합니다.
2,25453046,Hangong-Gan,menu,52,0.37254603936839203,True,푸주 마라 순살 닭구이
2,25453046,Hangong-Gan,menu,53,0.37246225797758126,True,부추 소금 순살 닭구이
2,25453046,Hangong-Gan,menu,85,0.3496130866840905,True,고추장 직화 순살 닭구이
3,6538765,루블랑,review,16,0.5376305770276215,None,"이 지하 를 자랑하는 세련된 된 조용한 맛있는 음식과. 양쪽에 있는 것이 바로 entrée 분할 및 점심 식사. 비싼 기대에 기반하지 않은 리뷰, 도. (안 싼, 한다면!) 다시 돌아올 겁니다!"


In [9]:
# 8. Weather MCP 원본 응답
from services.weather_mcp_client import get_weather_via_mcp

weather_query = (
    parsed_query.weather_request.query
    if parsed_query.weather_request
    else parsed_query.original_question
)
weather = await get_weather_via_mcp(
    weather_query,
    plan.origin_lat,
    plan.origin_lng,
    parsed_query.language,
    plan.location_name,
)
pprint(weather)

if not weather.get("available"):
    print("주의: MCP 서버 또는 기상청 API가 응답하지 않아 순위 변화가 없을 수 있습니다.")

{'available': True,
 'base_at': '2026-07-15T14:00:00+09:00',
 'condition': 'clear',
 'condition_label': '강수 없음',
 'feels_like': 'hot',
 'feels_like_label': '더움',
 'forecast_for': '2026-07-16T19:00:00+09:00',
 'forecast_offset_minutes': 0,
 'humidity_pct': 50.0,
 'is_forecast': True,
 'language': 'ko',
 'location': {'lat': 37.5568707448873,
              'lng': 126.923778562273,
              'nx': 59,
              'ny': 126},
 'place_name': '홍대입구역 2호선',
 'precipitation_probability_pct': 0.0,
 'rainfall_mm': 0.0,
 'recommendation_policy': '날씨는 보조 근거로만 사용하고, 사용자 필수 조건을 무시하거나 후보 데이터에 없는 시설을 '
                          '추측하지 마세요.',
 'requested_for': '2026-07-16T19:00:00+09:00',
 'resolved_query': '내일 저녁 홍대 날씨',
 'should_affect_recommendation': True,
 'sky': 'clear',
 'sky_label': '맑음',
 'source': 'KMA-vilage-forecast',
 'summary': '내일 저녁: 강수 없음, 28°C입니다.',
 'target_label': '내일 저녁',
 'temperature_c': 28.0,
 'usage_guidance': ['냉방이 잘 되는 실내 공간은 보조 장점이 될 수 있습니다.'],
 'weather_tags': ['HOT'],
 

In [10]:
# 9. Weather MCP 적용 후 전체 순위와 변경 이유
from services.weather_reranker import rerank_with_weather

weather_candidates = rerank_with_weather(
    deepcopy(raw_candidates),
    weather,
    parsed_query.original_question,
    source_mode="rag_mcp",
)

raw_rank_by_id = {c["restaurant_id"]: rank for rank, c in enumerate(raw_candidates, 1)}
raw_score_by_id = {c["restaurant_id"]: c["score"] for c in raw_candidates}

comparison_rows = []
for mcp_rank, c in enumerate(weather_candidates, 1):
    restaurant_id = c["restaurant_id"]
    raw_rank = raw_rank_by_id[restaurant_id]
    features = c.get("weather_features") or {}
    comparison_rows.append({
        "mcp_rank": mcp_rank,
        "rag_rank": raw_rank,
        "rank_change": raw_rank - mcp_rank,
        "restaurant_id": restaurant_id,
        "name": c["name"],
        "raw_rag_score": raw_score_by_id[restaurant_id],
        "weather_score": c.get("weather_score"),
        "final_score": c.get("score"),
        "weather_reasons": " | ".join(c.get("weather_reasons", [])),
        "distance_km": c.get("distance_km"),
        "parking": c.get("has_parking"),
        "outdoor_confidence": c.get("outdoor_confidence"),
        "warm_menu": features.get("has_warm_menu"),
        "warm_matches": ", ".join(features.get("warm_menu_matches", [])),
        "cool_menu": features.get("has_cool_menu"),
        "cool_matches": ", ".join(features.get("cool_menu_matches", [])),
    })

show_table(comparison_rows)

print("순위 상승: rank_change > 0")
print("순위 하락: rank_change < 0")

mcp_rank,rag_rank,rank_change,restaurant_id,name,raw_rag_score,weather_score,final_score,weather_reasons,distance_km,parking,outdoor_confidence,warm_menu,warm_matches,cool_menu,cool_matches
1,1,0,33405834,살롱 드 상상,0.10642024113732206,0.5,0.9,,1.1692589521459695,None,unknown,False,,False,
2,2,0,25453046,Hangong-Gan,0.10197725822568388,0.5,0.866600467248293,,0.5505386241093928,True,unknown,False,,False,
3,3,0,6538765,루블랑,0.09484203729245018,0.5,0.8129623934609834,,0.36205006973087395,True,unknown,False,,False,
4,4,0,9598657,랑빠스81,0.08984681665252393,0.5,0.7754114870804536,,0.38436111312037236,False,unknown,False,,False,
5,6,1,1139517,벽제 갈비,0.08399727182997938,0.65,0.7614383123533154,"더운 날 어울리는 시원한 메뉴가 있음 (평양냉면, 한우불고기냉면정식, 순메밀냉면)",1.2416434839847603,True,unknown,True,"한우갈비탕, 한우갈비양곰탕",True,"평양냉면, 한우불고기냉면정식, 순메밀냉면"
6,5,-1,8728506,미분당 신촌점,0.0852650178452127,0.5,0.7409684243070926,,0.864067545614737,False,unknown,False,,False,
7,7,0,25139751,코스모 더 케이브,0.07787079864323157,0.5,0.6853833655028014,,0.3844765878972961,None,unknown,False,,False,
8,8,0,26379212,온달집 홍대직영점,0.0777092242180033,0.5,0.6841687512639949,,0.7173604637217327,None,unknown,False,,False,
9,9,0,5979642,누나홀닭 홍대점,0.07563923863617554,0.5,0.6686079101329796,,0.11422970030416868,None,high,True,"해물누룽지탕, 해물짬뽕탕, 백합조개탕",False,
10,10,0,2170347,죠티인도레스토랑,0.07490927637106484,0.5,0.6631205159507486,,1.2680509415208472,None,unknown,False,,False,


순위 상승: rank_change > 0
순위 하락: rank_change < 0


In [ ]:
# 10. 실제로 순위가 바뀐 후보와 날씨 이유만 보기
changed_rows = [
    row for row in comparison_rows
    if row["rank_change"] != 0 or row["weather_reasons"]
]
show_table(changed_rows)

In [ ]:
# 11. 합성 날씨 시나리오 회귀 디버깅
# 실제 예보와 별개로 더움·추움·비·비+강풍 규칙이 과도하게 작동하는지 확인합니다.
SYNTHETIC_WEATHER_SCENARIOS = {
    "hot_30c": {
        "available": True, "condition": "clear", "temperature_c": 30.0, "wind_speed_mps": 2.0,
    },
    "cold_0c": {
        "available": True, "condition": "clear", "temperature_c": 0.0, "wind_speed_mps": 2.0,
    },
    "rain": {
        "available": True, "condition": "rain", "temperature_c": 18.0, "wind_speed_mps": 3.0,
    },
    "rain_strong_wind": {
        "available": True, "condition": "rain", "temperature_c": 18.0, "wind_speed_mps": 9.0,
    },
}

scenario_summary = []
scenario_reason_rows = []
original_ids = [c["restaurant_id"] for c in raw_candidates]
original_rank = {restaurant_id: rank for rank, restaurant_id in enumerate(original_ids, 1)}

for scenario_name, scenario_weather in SYNTHETIC_WEATHER_SCENARIOS.items():
    reranked = rerank_with_weather(
        deepcopy(raw_candidates), scenario_weather,
        parsed_query.original_question, source_mode="rag_mcp",
    )
    changes = [
        original_rank[c["restaurant_id"]] - rank
        for rank, c in enumerate(reranked, 1)
    ]
    scenario_summary.append({
        "scenario": scenario_name,
        "candidates": len(reranked),
        "changed_candidates": sum(change != 0 for change in changes),
        "largest_rise": max(changes, default=0),
        "largest_drop": min(changes, default=0),
        "candidates_with_reasons": sum(bool(c.get("weather_reasons")) for c in reranked),
        "top5": " | ".join(c["name"] for c in reranked[:5]),
    })
    for rank, c in enumerate(reranked, 1):
        if c.get("weather_reasons"):
            scenario_reason_rows.append({
                "scenario": scenario_name,
                "mcp_rank": rank,
                "rag_rank": original_rank[c["restaurant_id"]],
                "rank_change": original_rank[c["restaurant_id"]] - rank,
                "name": c["name"],
                "weather_score": c.get("weather_score"),
                "reasons": " | ".join(c.get("weather_reasons", [])),
            })

show_table(scenario_summary)
show_table(scenario_reason_rows)

In [ ]:
# 12. 특정 후보의 모든 원본 값을 상세 확인
# 보고 싶은 순위 또는 restaurant_id로 바꾸세요.
DEBUG_MCP_RANK = 1
DEBUG_RESTAURANT_ID = None

if DEBUG_RESTAURANT_ID is None:
    selected = weather_candidates[DEBUG_MCP_RANK - 1]
else:
    selected = next(c for c in weather_candidates if c["restaurant_id"] == DEBUG_RESTAURANT_ID)

pprint(selected, sort_dicts=False)

In [ ]:
# 13. RAG_ONLY 안전성 확인: 날씨 조회/재랭킹 없이 원래 순서가 유지되어야 함
from services.weather_reranker import prepare_rag_only_candidates

rag_only_candidates = prepare_rag_only_candidates(deepcopy(raw_candidates))
assert [c["restaurant_id"] for c in rag_only_candidates] == [c["restaurant_id"] for c in raw_candidates]
assert all(c.get("weather_score") is None for c in rag_only_candidates)
assert all(not c.get("weather_reasons") for c in rag_only_candidates)
print("RAG_ONLY 순서 보존 및 날씨 값 제거 확인 완료")

In [ ]:
# 14. 선택 사항: 최종 GPT 답변까지 생성
# RAG/MCP 디버깅만 할 때는 False로 두세요.
RUN_FINAL_GPT = False

if RUN_FINAL_GPT:
    from services.llm import generate_recommendation
    final_answer = generate_recommendation(
        parsed_query.original_question,
        parsed_query.language,
        weather_candidates[:10],
        weather,
        source_mode="rag_mcp",
    )
    print(final_answer)
else:
    print("최종 GPT 호출 생략: RUN_FINAL_GPT=True로 바꾸면 실행됩니다.")

In [ ]:
# 15. 선택 사항: 이번 디버그 결과를 JSON으로 저장
SAVE_TRACE = False
TRACE_PATH = BACKEND_DIR / "debug_outputs" / "rag_mcp_trace.json"

if SAVE_TRACE:
    import json
    TRACE_PATH.parent.mkdir(parents=True, exist_ok=True)
    trace = {
        "question_case": QUESTION_CASE,
        "search_plan": {key: str(value) if key == "target_visit_at" else value for key, value in asdict(plan).items()},
        "rag_result_meta": {key: value for key, value in rag_result.items() if key != "candidates"},
        "rag_candidates": raw_candidates,
        "weather": weather,
        "weather_reranked_candidates": weather_candidates,
    }
    TRACE_PATH.write_text(json.dumps(trace, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print("saved:", TRACE_PATH)
else:
    print("저장하지 않음: SAVE_TRACE=True로 바꾸면 JSON trace가 생성됩니다.")